# LangChain + Pinecone: Vector Store Basics

Pinecone version of the Chroma notebook — same IPL dataset, same LangChain `VectorStore` interface, but a cloud-hosted index instead of a local on-disk DB.

Requires:
- `PINECONE_API_KEY` and `GOOGLE_API_KEY` set in your `.env`
- A Pinecone account (free tier is fine) — https://app.pinecone.io

Key differences from Chroma you'll see below:
- You must **explicitly create the index** first, with a dimension matching your embedding model
- No `persist_directory` / `collection_name` — Pinecone is always server-side
- No `update_document()` — Pinecone updates are just re-`add_documents()` with the same ID (upsert)
- No `vector_store.get()` — listing/fetching by ID works differently (via the raw `index` object)

In [1]:
from langchain_core.documents import Document
from langchain_pinecone import PineconeVectorStore
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from pinecone import Pinecone, ServerlessSpec
from dotenv import load_dotenv

load_dotenv()

c:\Users\poora\Desktop\work\Langchain\langenv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


True

## Create the Pinecone index

Unlike Chroma (which creates a collection automatically), Pinecone needs the index created explicitly, with a `dimension` that matches your embedding model's output size.

`gemini-embedding-001` outputs **3072** dimensions by default — same value we saw in the Chroma notebook's `embeddings` array shape `(5, 3072)`.

In [2]:
pc = Pinecone()  # reads PINECONE_API_KEY from environment automatically

index_name = "sample"

if not pc.has_index(index_name):
    pc.create_index(
        name=index_name,
        dimension=3072,
        metric="cosine",
        spec=ServerlessSpec(cloud="aws", region="us-east-1")
    )

index = pc.Index(index_name)

## Create sample documents

In [3]:
# Create LangChain documents for IPL players

doc1 = Document(
    page_content="Virat Kohli is one of the most successful and consistent batsmen in IPL history. Known for his aggressive batting style and fitness, he has led the Royal Challengers Bangalore in multiple seasons.",
    metadata={"team": "Royal Challengers Bangalore"}
)
doc2 = Document(
    page_content="Rohit Sharma is the most successful captain in IPL history, leading Mumbai Indians to five titles. He's known for his calm demeanor and ability to play big innings under pressure.",
    metadata={"team": "Mumbai Indians"}
)
doc3 = Document(
    page_content="MS Dhoni, famously known as Captain Cool, has led Chennai Super Kings to multiple IPL titles. His finishing skills, wicketkeeping, and leadership are legendary.",
    metadata={"team": "Chennai Super Kings"}
)
doc4 = Document(
    page_content="Jasprit Bumrah is considered one of the best fast bowlers in T20 cricket. Playing for Mumbai Indians, he is known for his yorkers and death-over expertise.",
    metadata={"team": "Mumbai Indians"}
)
doc5 = Document(
    page_content="Ravindra Jadeja is a dynamic all-rounder who contributes with both bat and ball. Representing Chennai Super Kings, his quick fielding and match-winning performances make him a key player.",
    metadata={"team": "Chennai Super Kings"}
)

docs = [doc1, doc2, doc3, doc4, doc5]

# Pinecone needs explicit string IDs for update/delete later (unlike Chroma, which auto-generates UUIDs)
doc_ids = ["doc1", "doc2", "doc3", "doc4", "doc5"]

## Create the vector store

`gemini-embedding-001` is the current stable Gemini text embedding model (the older `embedding-001` and `text-embedding-004` models are both deprecated).

In [4]:
embeddings = GoogleGenerativeAIEmbeddings(model="gemini-embedding-001")

vector_store = PineconeVectorStore(
    index=index,
    embedding=embeddings
)

## Add documents

Passing explicit `ids` here (instead of letting them auto-generate) makes update/delete predictable later — Pinecone has no `get()` to look IDs up afterward like Chroma does.

In [5]:
# add documents
vector_store.add_documents(docs, ids=doc_ids)

['doc1', 'doc2', 'doc3', 'doc4', 'doc5']

## View documents

Pinecone has no `vector_store.get()` equivalent to dump everything at once. To check what's in the index, fetch by known ID via the raw `index` object, or check overall index stats.

In [6]:
# index-level stats (count, dimension, etc.) — closest equivalent to Chroma's full dump
index.describe_index_stats()

{'dimension': 3072,
 'index_fullness': 0.0,
 'metric': 'cosine',
 'namespaces': {'': {'vector_count': 5}},
 'total_vector_count': 5,
 'vector_type': 'dense'}

In [7]:
# fetch specific documents by ID
index.fetch(ids=doc_ids)

FetchResponse(namespace='', vectors={'doc2': Vector(id='doc2', values=[-0.0198972058, 0.0109271351, 0.0173060261, -0.0674819872, 0.000747970422, 0.00626170635, -0.00577525934, 0.0242827088, -0.0289491396, 0.00891696475, -0.00651673321, -0.0119616836, -0.00101985491, 0.0459609106, 0.115392022, -0.00331223826, -0.0157784056, 0.00247745449, 0.0123895006, -0.00105018052, 0.00578086916, -0.00533109158, 0.00873262435, -0.0167614836, -0.0154290935, -0.0111460509, 0.00701568834, -0.00945331249, 0.0305102468, -0.0125521673, 0.00636907667, -0.0174735133, 0.0233901385, 0.0171180591, -0.00925425068, 0.00884752441, 0.01743735, 0.0208273605, -0.0145800989, 0.00721100671, -0.0147977956, -0.00725857, -0.0035207, -0.0233879201, -0.00105270592, -0.00835015252, 0.0111192735, -0.0317656025, -0.0253255423, 0.00181313208, -0.00190290925, -0.0100730471, -0.0210014805, -0.200855464, 0.0190751571, 0.020310903, -0.0140627036, -0.00776316551, 0.024222089, 0.012181093, -0.000826055184, 0.023090614, -0.015710799, 

## Similarity search

In [8]:
# search documents
vector_store.similarity_search(
    query='Who among these are a bowler?',
    k=2
)

[Document(id='doc4', metadata={'team': 'Mumbai Indians'}, page_content='Jasprit Bumrah is considered one of the best fast bowlers in T20 cricket. Playing for Mumbai Indians, he is known for his yorkers and death-over expertise.'),
 Document(id='doc5', metadata={'team': 'Chennai Super Kings'}, page_content='Ravindra Jadeja is a dynamic all-rounder who contributes with both bat and ball. Representing Chennai Super Kings, his quick fielding and match-winning performances make him a key player.')]

In [9]:
# search with similarity score
vector_store.similarity_search_with_score(
    query='Who among these are a bowler?',
    k=2
)

[(Document(id='doc4', metadata={'team': 'Mumbai Indians'}, page_content='Jasprit Bumrah is considered one of the best fast bowlers in T20 cricket. Playing for Mumbai Indians, he is known for his yorkers and death-over expertise.'),
  0.680763245),
 (Document(id='doc5', metadata={'team': 'Chennai Super Kings'}, page_content='Ravindra Jadeja is a dynamic all-rounder who contributes with both bat and ball. Representing Chennai Super Kings, his quick fielding and match-winning performances make him a key player.'),
  0.669998169)]

## Metadata filtering

Same idea as Chroma, but Pinecone's filter syntax uses Mongo-style operators (e.g. `$eq`) rather than a plain dict key-value match. A plain `{"team": "..."}` dict also works as shorthand for equality.

In [10]:
vector_store.similarity_search_with_score(
    query="Chennai Super Kings player",
    k=5,
    filter={"team": {"$eq": "Chennai Super Kings"}}
)

[(Document(id='doc5', metadata={'team': 'Chennai Super Kings'}, page_content='Ravindra Jadeja is a dynamic all-rounder who contributes with both bat and ball. Representing Chennai Super Kings, his quick fielding and match-winning performances make him a key player.'),
  0.756324828),
 (Document(id='doc3', metadata={'team': 'Chennai Super Kings'}, page_content='MS Dhoni, famously known as Captain Cool, has led Chennai Super Kings to multiple IPL titles. His finishing skills, wicketkeeping, and leadership are legendary.'),
  0.72391516)]

## Update a document

Pinecone has no `update_document()` method. An "update" is just re-adding a document with the **same ID** — Pinecone treats this as an upsert (overwrite) rather than an error or a duplicate.

In [11]:
# update documents (upsert by reusing the same id: "doc1")
updated_doc1 = Document(
    page_content="Virat Kohli, the former captain of Royal Challengers Bangalore (RCB), is renowned for his aggressive leadership and consistent batting performances. He holds the record for the most runs in IPL history, including multiple centuries in a single season. Despite RCB not winning an IPL title under his captaincy, Kohli's passion and fitness set a benchmark for the league. His ability to chase targets and anchor innings has made him one of the most dependable players in T20 cricket.",
    metadata={"team": "Royal Challengers Bangalore"}
)

vector_store.add_documents([updated_doc1], ids=["doc1"])

['doc1']

In [12]:
# confirm the update
index.fetch(ids=["doc1"])

FetchResponse(namespace='', vectors={'doc1': Vector(id='doc1', values=[-0.00719311, 0.0202993155, 0.0283715762, -0.0596735775, 0.00774613628, -0.00604558736, -0.00374005083, 0.014580681, -0.0409464352, 0.0054760715, 0.00395009108, -0.0168068111, -0.00250633014, 0.0754053071, 0.0991745368, -0.00174816733, 0.0157769397, -0.00634532, 0.0014657604, -0.0113716032, 0.000325534, -0.00943437591, 0.0122254966, -0.0112772901, -0.00680001732, -0.0202933475, 0.0234861113, 0.0153046567, 0.0115191713, -0.000789029, -0.0167573355, -0.0184999444, 0.0175973512, 0.0150574511, -0.00943118241, -0.00167904631, 0.00905044097, 0.0232152604, 0.00641091168, 0.0126434276, -0.00550697651, -0.000554566737, 0.00511947647, -0.0219694227, 0.0111598074, -0.0266875792, 0.0129754553, -0.0195173081, -0.00426431326, -0.0046653254, -0.000269171753, -0.00654596323, -0.0199412685, -0.203568831, 0.0023789613, 0.00855498202, -0.00548920827, 0.00763908634, 0.0224663578, 0.00801001, 0.02119877, 0.0289468449, -0.0209407583, -0.0

## Delete a document

In [13]:
# delete document by id
vector_store.delete(ids=["doc1"])

In [14]:
# confirm deletion — stats should now show one less vector
index.describe_index_stats()

{'dimension': 3072,
 'index_fullness': 0.0,
 'metric': 'cosine',
 'namespaces': {'': {'vector_count': 4}},
 'total_vector_count': 4,
 'vector_type': 'dense'}